In [ ]:
import openmc
import numpy as np
import openmc.deplete
from pathlib import Path
import structural_material_analysis as sma
import tokamak_radiation_environment as tre

In [ ]:
cross_sections_path = r'/home/davide/openmc_models/CROSS_SECTIONS/endfb-viii.0-hdf5/cross_sections.xml'
openmc.config['cross_sections'] = cross_sections_path
# users might want to change these to use specific xml files to use particular decay data or transport cross sections
chain_file = r'/home/davide/openmc_models/CROSS_SECTIONS/decay/ENDFB_VII_1_Chain/chain_endfb71_sfr.xml'
openmc.config['chain_file'] = chain_file

In [ ]:
inner_vessel_material = sma.materials.inconel718

In [ ]:
# Volumes of regions expressed in the unit of cm3 (from SolidWorks)
# Volumes of real geometry
INNER_volume = 181437118.09
FW_volume = 352170.38
STR1_volume = 3529943.32
FLiBe1_volume = 7104771.83
NM_volume = 3574791.48
STR2_volume = 10813774.60
FLiBe2_volume = 350171938.29
STR3_volume = 14002357.77

fusion_power = 525e6  # units Watts
energy_per_fusion_reaction = 17.6e6  # units eV
eV_to_Joules = 1.60218e-19  # multiplication factor to convert eV to Joules
number_of_neutrons_per_second = fusion_power / (energy_per_fusion_reaction * eV_to_Joules)
print('Number of neutrons per second', number_of_neutrons_per_second, '\n')

In [ ]:
inner_vessel_material.depletable = True
inner_vessel_material.volume = STR1_volume

In [ ]:
dt_plasma = sma.materials.dt_plasma
tungsten = sma.materials.tungsten
flibe = sma.materials.flibe
beryllium = sma.materials.beryllium
boroncarbide = sma.materials.boroncarbide
inconel718 = sma.materials.inconel718

materials = openmc.Materials([
    dt_plasma,
    tungsten,
    inner_vessel_material,
    flibe, 
    boroncarbide, 
    beryllium, 
    ])

(root, geometry) = sma.geometry.reactor_geometry(dt_plasma, tungsten, inner_vessel_material, flibe, beryllium, inconel718, boroncarbide)

############################################################################
# Define Settings
source = openmc.IndependentSource()
source.particle = "neutron"
radius = openmc.stats.Discrete([285], [1])
z_values = openmc.stats.Discrete([0], [1])
angle = openmc.stats.Uniform(a=np.radians(-10), b=np.radians(10))
source.space = openmc.stats.CylindricalIndependent(
    r=radius, phi=angle, z=z_values, origin=(0.0, 0.0, 0.0)
)
source.angle = openmc.stats.Isotropic()
source.energy = openmc.stats.muir(e0=14080000.0, m_rat=5.0, kt=20000.0)

settings = openmc.Settings()
settings.batches = 100
settings.particles = 10000
settings.inactive = 0
settings.run_mode = "fixed source"
settings.source = source
settings.photon_transport = True  # This line is required to switch on photons tracking
settings.output = {"tallies": False}

model = openmc.Model(geometry, materials, settings)

In [ ]:
operator = openmc.deplete.CoupledOperator(
    model=model,
    normalization_mode="source-rate",  # set for fixed source simulation, otherwise defaults to fission simulation
    reduce_chain=True,  # reduced to only the isotopes present in depletable materials and their possible progeny
    reduce_chain_level=20,
)

In [ ]:
# We define timesteps together with the source rate to make it clearer
timesteps_and_source_rates = [
    (31536000, number_of_neutrons_per_second),
    (1, 0),
    (4, 0),
    (25, 0),
    (30, 0),
    (60, 0),
    (180, 0),
    (300, 0),
    (1200, 0),
    (1800, 0),
    (3600, 0),
    (14400, 0),
    (21600, 0),
    (43200, 0),
    (86400, 0),
    (172800, 0),
    (259200, 0),
    (604800, 0),
    (1468800, 0),
    (2678400, 0),
    (10627200, 0),
    (15897600, 0),
    (31536000, 0),
    (63072000, 0),
    (63072000, 0),
    (63072000, 0),
    (63072000, 0),
    (63072000, 0),   
    (63072000, 0),
    (63072000, 0),    
    (63072000, 0),
    (157680000, 0),
    (157680000, 0),
    (157680000, 0),
    (157680000, 0),
    (157680000, 0),
    (157680000, 0),
    (315360000, 0),
    (315360000, 0),
    (315360000, 0),
    (315360000, 0),
    (315360000, 0),
    (788400000, 0),
    (788400000, 0),
    (788400000, 0),
    (788400000, 0),
    (788400000, 0),
    (1576800000, 0),
    (1576800000, 0),
]

# Uses list Python comprehension to get the timesteps and source_rates separately
timesteps = [item[0] for item in timesteps_and_source_rates]
source_rates = [item[1] for item in timesteps_and_source_rates]


# PredictorIntegrator has been selected as the depletion operator for this example as it is a fast first order Integrator
# OpenMC offers several time-integration algorithms https://docs.openmc.org/en/stable/pythonapi/deplete.html#primary-api\n",
# CF4Integrator should normally be selected as it appears to be the most accurate https://dspace.mit.edu/handle/1721.1/113721\n",
integrator = openmc.deplete.PredictorIntegrator(
    operator=operator,
    timesteps=timesteps,
    source_rates=source_rates
)

In [ ]:
integrator.integrate()


In [ ]:
results = openmc.deplete.ResultsList.from_hdf5("depletion_results.h5")

times, number_of_Co58_atoms = results.get_atoms(inner_vessel_material, "Co58")

for time, num in zip(times, number_of_Co58_atoms):
    print(f" Time {time}s. Number of Co58 atoms {num}")

In [ ]:
time, activities = results.get_activity(inner_vessel_material, units='Bq', by_nuclide=True)

In [ ]:
s_year = 365.25 * 24 * 3600
kg_str1 = inner_vessel_material.density * STR1_volume *1e-3
kg_str1 = inner_vessel_material.density * STR1_volume *1e-3

In [ ]:
activities[1]

In [ ]:
import matplotlib.pyplot as plt

# Usa il tema seaborn per un aspetto più bello
plt.style.use('default')

# Calcola ciascun valore di decay heat 

fig = plt.figure(figsize=(16, 12))

# Cambia il colore e lo stile delle linee
plt.loglog(time_steps[:]/s_year-1, attività_divise, 'b-', linewidth=3, label='OpenMC')

plt.xlabel('Time after shutdown [years]', fontsize=24)
plt.ylabel('Activity [Bq/kg]', fontsize=24)
# plt.title('Decay heat with time', fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Aggiungi una griglia
plt.grid(True, which="both", ls="--", linewidth=0.5)

# Aggiungi una legenda
plt.legend(fontsize=16)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Usa il tema seaborn per un aspetto più bello
plt.style.use('default')

fig = plt.figure(figsize=(16, 12))

# Cambia il colore e lo stile delle linee
plt.loglog(time_steps[:]/s_year-1, calore_diviso, 'b-', linewidth=3, label='OpenMC')
# plt.loglog(tempo, attivita_d, color='pink', alpha=0.3, linewidth=2, label='FISPACT-II divertor')
# plt.loglog(tempo, attivita_e, color='pink', alpha=0.3, linewidth=2, label='FISPACT-II external')
# Colora l'area tra attivita_d e attivita_e

plt.xlabel('Time after shutdown [years]', fontsize=24)
plt.ylabel('Heat Output [kW/kg]', fontsize=24)
# plt.title('Decay heat with time', fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# Aggiungi una griglia
plt.grid(True, which="both", ls="--", linewidth=0.5)

# Aggiungi una legenda
plt.legend(fontsize=16)
ax = plt.gca()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.show()